A Gentle Appeal : 

"If this Notebook / robust pipeline (especially the XGBoost early-stopping fix) helps you or improves your score / Coding skills , please consider upvoting! It helps me track community utility and advance on Kaggle. Thank you!"

In [ ]:
# U1

# - Refinement to Mid-Level Production Code (FIXED TypeError)

# Forked Code - Simplified Pipeline and Better Benchmarks

# Date of Refinement: 28 October 2025

# --- 1. Import Libraries ---

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge # Using Ridge instead of simple Linear Regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor 

# --- Configuration ---

# NOTE: This directory path is specific to a Kaggle environment and must be adjusted locally.

DATA_DIRECTORY = '/kaggle/input/house-prices-advanced-regression-techniques/'
TARGET = 'SalePrice'
RANDOM_STATE = 42 # Standard seed for reproducibility
# SUBMISSION_FILE_NAME = 'submission_mid_level_xgb.csv' # Define the output file name
SUBMISSION_FILE_NAME = 'submission.csv' # Use the standard file name

# --- 2. DATA LOADING & INITIAL PREP (MOCK) ---

try:
    # Attempt to load data from the configured path
    df_train = pd.read_csv(DATA_DIRECTORY + 'train.csv')
    df_test = pd.read_csv(DATA_DIRECTORY + 'test.csv') 
except FileNotFoundError:
    # If not in the Kaggle environment, assume the files are in the local directory
    # NOTE: You must have 'train.csv' and 'test.csv' in the same folder as this script.
    print("WARNING: Files not found in Kaggle path. Attempting to load from local directory.")
    DATA_DIRECTORY = './'
    try:
        df_train = pd.read_csv(DATA_DIRECTORY + 'train.csv')
        df_test = pd.read_csv(DATA_DIRECTORY + 'test.csv') 
    except FileNotFoundError:
        print("FATAL ERROR: Files not found locally either. Please check file paths.")
        # Exit gracefully if data cannot be loaded
        import sys
        sys.exit(1)


# --- 3. TARGET & OUTLIER HANDLING ---
# Remove severe outliers (critical step for linear models)

df_train_cleaned = df_train[df_train['GrLivArea'] < 4000].copy()

# Log transformation of the target (standard practice for SalePrice)

y = np.log(df_train_cleaned[TARGET])
X = df_train_cleaned.drop(TARGET, axis=1)

# --- 4. FEATURE DEFINITION (Mid-Level Robust Set) ---

# A reasonable feature set, expanded from the minimal list

NUMERIC_FEATURES = [
    'OverallQual', 'GrLivArea', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 
    'FullBath', 'YearBuilt', 'LotFrontage' 
]
CATEGORICAL_FEATURES = [
    'MSZoning', 'Neighborhood', 'HouseStyle', 'RoofStyle', 
    'Foundation', 'ExterQual', 'KitchenQual', 'YrSold' 
]

# Clean up features not present in the combined set (due to data quality/errors)
# This handles cases where columns might be missing between train and test/validation splits

NUMERIC_FEATURES = [f for f in NUMERIC_FEATURES if f in X.columns]
CATEGORICAL_FEATURES = [f for f in CATEGORICAL_FEATURES if f in X.columns]

# Combine for the final feature list

FINAL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES


# --- 5. FULL PREPROCESSING PIPELINE ---

# Numerical Imputation (Median) -> Scaling

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()) 
])

# Categorical Imputation ('Missing') -> One-Hot Encoding

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')), 
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# Column Transformer: Applies different transformations to different columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUMERIC_FEATURES),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES)
    ],
    remainder='passthrough' # Pass through any unlisted columns
)


# --- 6. SPLIT DATA ---
# Use the defined features for the training/validation split (for internal evaluation only)

X_train, X_val, y_train, y_val = train_test_split(
    X[FINAL_FEATURES], # Select only the defined features
    y, 
    test_size=0.2, 
    random_state=RANDOM_STATE
)


# --- 7. MODELING (Integrated Pipelines and Benchmarks) ---

print(f"--- MID-LEVEL MODEL EVALUATION (Features: {len(FINAL_FEATURES)}) ---")

# Define three integrated pipelines

models = {
    "Ridge Regression": Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', Ridge(alpha=10.0, random_state=RANDOM_STATE, solver='lsqr')) 
    ]),
    "Random Forest": Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(
            n_estimators=100, 
            max_depth=10, 
            random_state=RANDOM_STATE, 
            n_jobs=-1
        ))
    ]),
    "XGBoost": Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', XGBRegressor(
            n_estimators=500, 
            learning_rate=0.05, 
            max_depth=5, 
            random_state=RANDOM_STATE, 
            n_jobs=-1
        ))
    ])
}

# Train and evaluate all models

best_rmse = float('inf')
best_model_name = None

for name, model in models.items():
    # Fit the entire pipeline
    model.fit(X_train, y_train)
    
    # Predict and calculate RMSE
    predictions = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    
    # Track the best model
    if rmse < best_rmse:
        best_rmse = rmse
        best_model_name = name

    # Print results
    print(f"{name} RMSE (log): {rmse:.5f}")

# --- 8. FINAL MODEL TRAINING AND PREDICTION ---

print("\n--- FINAL TRAINING & PREDICTION FOR SUBMISSION ---")

# 1. Select the best performing model (XGBoost is often the best default)
# For simplicity, we choose XGBoost, but in a real scenario, use the 'best_model_name'
final_model = models["XGBoost"]

# 2. Re-train the chosen model on the *entire* training dataset (X and y)
print(f"Training final model ({final_model.steps[-1][0]})...")
final_model.fit(X[FINAL_FEATURES], y) # Use the entire cleaned training data

# 3. Predict on the test data
# Note: Ensure test data only contains the features used for training
X_test = df_test[FINAL_FEATURES].copy()
log_predictions = final_model.predict(X_test)

# 4. Inverse-transform the predictions (log -> exponential)
# The prediction is log(SalePrice), so we use np.exp() to get SalePrice
final_predictions = np.exp(log_predictions)
print("Prediction complete. Predictions inverse-transformed.")




🧠 Model Performance Context : 

The   submitted model was the XGBoost Regressor, trained on the full dataset, which achieved the following RMSE (log) on our internal validation set:

ModelInternal Validation : 

RMSE (log)Ridge Regression0.14800

Random Forest0.16493

XGBoost0.15612

Key Observation:The Ridge Regression model had the best internal validation score (0.14800).
The XGBoost model had an internal validation score of 0.15612, which is very close to its public leaderboard score of 0.15566.
This suggests that our XGBoost pipeline is robust and generalizes well to the unseen test data.

In [ ]:
# U2

# notebook 2 # U2 - Refinement to Mid-Level Production Code (FIXED TypeError)

# Forked Code - Simplified Pipeline and Better Benchmarks

# Date of Refinement: 29 October 2025

# --- 1. Import Libraries ---

import pandas as pd
import numpy as np
import os
import sys

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# --- 2. Configuration & Constants ---

DATA_DIRECTORY = '/kaggle/input/house-prices-advanced-regression-techniques/'
TARGET = 'SalePrice'
RANDOM_STATE = 42 # Reproducibility seed
SUBMISSION_FILE_NAME = 'submission.csv'

# Define the feature set for a robust mid-level model
NUMERIC_FEATURES = [
    'OverallQual', 'GrLivArea', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
    'FullBath', 'YearBuilt', 'LotFrontage'
]
CATEGORICAL_FEATURES = [
    'MSZoning', 'Neighborhood', 'HouseStyle', 'RoofStyle',
    'Foundation', 'ExterQual', 'KitchenQual', 'YrSold'
]

# --- 3. Data Loading Utility ---

def load_data(data_path):
    """Loads train and test data, handling different environments."""
    try:
        df_train = pd.read_csv(data_path + 'train.csv')
        df_test = pd.read_csv(data_path + 'test.csv')
    except FileNotFoundError:
        print("🚨 WARNING: Files not found in configured path. Attempting to load from local directory.")
        try:
            df_train = pd.read_csv('train.csv')
            df_test = pd.read_csv('test.csv')
        except FileNotFoundError:
            print("FATAL ERROR: Data files not found locally either. Exiting.")
            sys.exit(1)
            
    print(f"Data loaded: Train rows = {len(df_train)}, Test rows = {len(df_test)}")
    return df_train, df_test

# --- 4. Main Execution Block ---

if __name__ == '__main__':
    
    # Load data
    df_train, df_test = load_data(DATA_DIRECTORY)

    # --- 4.1. TARGET & OUTLIER HANDLING ---
    
    # Remove severe outliers based on GrLivArea (critical for linear models)
    df_train_cleaned = df_train[df_train['GrLivArea'] < 4000].copy()
    
    # Apply log transformation to the target variable
    y_full = np.log(df_train_cleaned[TARGET])
    X_full = df_train_cleaned.drop(TARGET, axis=1)

    # Filter features based on columns present in the training data
    final_numeric = [f for f in NUMERIC_FEATURES if f in X_full.columns]
    final_categorical = [f for f in CATEGORICAL_FEATURES if f in X_full.columns]
    FINAL_FEATURES = final_numeric + final_categorical

    # Select only the defined features
    X_full = X_full[FINAL_FEATURES]
    
    # --- 4.2. PREPROCESSING PIPELINE SETUP ---

    # Pipeline for Numerical Features: Impute Median, then Scale
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Pipeline for Categorical Features: Impute 'Missing', then One-Hot Encode
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
    ])

    # Column Transformer: Applies different transformations to different columns
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, final_numeric),
            ('cat', categorical_transformer, final_categorical)
        ],
        remainder='passthrough' # Keep any unlisted columns (though none are expected here)
    )

    # --- 4.3. SPLIT DATA ---
    
    # Split the cleaned data into training and validation sets for internal RMSE check
    X_train, X_val, y_train, y_val = train_test_split(
        X_full, y_full, test_size=0.2, random_state=RANDOM_STATE
    )

# --- 7. MODELING (Integrated Pipelines and Benchmarks) ---

    print(f"\n--- MID-LEVEL MODEL EVALUATION (Features: {len(FINAL_FEATURES)}) ---")

    # Define integrated pipelines for comparison
    models = {
        "Ridge Regression": Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('regressor', Ridge(alpha=10.0, random_state=RANDOM_STATE, solver='lsqr'))
        ]),
        "Random Forest": Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('regressor', RandomForestRegressor(
                n_estimators=100,
                max_depth=10,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ]),
        "XGBoost": Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('regressor', XGBRegressor(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=5,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ])
    }

    # Train and evaluate all models
    best_rmse = float('inf')
    best_model_name = None

    for name, model in models.items():
        # Fit the entire pipeline on the training data
        model.fit(X_train, y_train)
        
        # Predict on the validation set and calculate RMSE
        predictions = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, predictions))
        
        # Track the best model
        if rmse < best_rmse:
            best_rmse = rmse
            best_model_name = name

        print(f"| {name:<20} | RMSE (log): {rmse:.5f} |")

# --- 8. FINAL MODEL TRAINING AND PREDICTION ---

    print("\n--- FINAL TRAINING & PREDICTION FOR SUBMISSION ---")
    
    # 1. Select the final model (often XGBoost or the best_model_name)
    final_model = models["XGBoost"]

    # 2. Re-train the chosen model on the *entire* training dataset (X_full and y_full)
    print(f"✅ Training final model: {final_model.steps[-1][0]} on ALL data...")
    final_model.fit(X_full, y_full)
    
    # 3. Predict on the test data
    # Select only the defined features from the test set
    X_test_final = df_test[FINAL_FEATURES].copy()
    log_predictions = final_model.predict(X_test_final)

    # 4. Inverse-transform the predictions: e^x to get the actual SalePrice
    final_predictions = np.exp(log_predictions)
    print("Prediction complete. Results inverse-transformed.")


# --- 9. SUBMISSION FILE GENERATION ---

    # 1. Create the submission DataFrame
    submission_df = pd.DataFrame({
        'Id': df_test['Id'], # Use the original Id column from the test data
        'SalePrice': final_predictions
    })

    # 2. Write the DataFrame to a CSV file
    submission_df.to_csv(SUBMISSION_FILE_NAME, index=False)

    print(f"\n🎉 Submission file generated successfully: {SUBMISSION_FILE_NAME}")
    print(f"Total predictions: {len(submission_df)}")

Features - 17:

The script used a total of 17 cleaned and engineered features (a combination of numeric and one-hot encoded categorical features) for training.

RMSE (log): This is the primary performance metric, the Root Mean Squared Error, calculated on the log-transformed SalePrice. A lower RMSE value indicates a better model fit.

Ridge Regression: 0.14800: This linear model performed the best out of the three on the validation data.

XGBoost: 0.15612: This boosting model performed slightly worse than Ridge in this specific split/configuration.

Random Forest: 0.16493: This ensemble model performed the worst.

In [ ]:
# U3 - Enhanced Code (Moving from Midlevel Benchmarks to Robust Modeling)

# Forked Code - Enhanced for Performance

# Date of Enhancement: 30 October 2025

# --- 1. Import Libraries ---

import pandas as pd
import numpy as np
import os
import datetime
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge # Ridge is generally better than simple Linear Regression for House Prices
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler # Added StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor 

# --- Configuration (Standard) ---
DATA_DIRECTORY = '/kaggle/input/house-prices-advanced-regression-techniques/'
TARGET = 'SalePrice'
RANDOM_STATE = 42 # Standard seed for reproducibility

# --- 2. DATA LOADING & INITIAL PREP ---
try:
    df_train = pd.read_csv(DATA_DIRECTORY + 'train.csv')
    df_test = pd.read_csv(DATA_DIRECTORY + 'test.csv') # Load test data for consistent preprocessing
except FileNotFoundError:
    print("FATAL ERROR: Files not found. Check DATA_DIRECTORY path.")
    raise

# Combine for consistent feature engineering/imputation (Crucial step)
df_full = pd.concat([df_train.drop(TARGET, axis=1), df_test], ignore_index=True)

# --- 3. CRITICAL 'TRIMMING' (Outlier Removal) ---
# Apply trimming before combining, only on the training set.
df_train_cleaned = df_train[df_train['GrLivArea'] < 4000].copy()
y = np.log(df_train_cleaned[TARGET])
X = df_train_cleaned.drop(TARGET, axis=1)

# Recalculate full_df based on the cleaned train set and the original test set
df_full_cleaned = pd.concat([X, df_test], ignore_index=True)


# --- 4. FEATURE ENGINEERING (Advanced) ---

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Creates derived and categorical features for better model performance."""
    df_eng = df.copy()
    
    # 1. Total Square Footage
    df_eng['TotalSF'] = df_eng['TotalBsmtSF'] + df_eng['1stFlrSF'] + df_eng['2ndFlrSF']
    
    # 2. Year Features
    df_eng['YearsSinceRemod'] = df_eng['YrSold'] - df_eng['YearRemodAdd']
    df_eng['Age'] = df_eng['YrSold'] - df_eng['YearBuilt']
    df_eng.loc[df_eng['Age'] < 0, 'Age'] = 0 # Safety check
    
    # 3. Simple Interaction Term
    df_eng['OverallQual_GrLivArea'] = df_eng['OverallQual'] * df_eng['GrLivArea']
    
    # 4. Binary Features (Has feature or not)
    df_eng['HasBasement'] = (df_eng['TotalBsmtSF'] > 0).astype(int)
    df_eng['HasGarage'] = (df_eng['GarageArea'] > 0).astype(int)
    
    # NOTE: Many missing values are 'None' (meaning no feature), so they should be imputed 0 or 'None' as appropriate
    
    return df_eng

df_full_features = engineer_features(df_full_cleaned.drop('Id', axis=1))


# --- 5. ROBUST FEATURE SET & PREPROCESSING PIPELINE ---

# Expanded feature set for better predictive power
ROBUST_FEATURES = [
    'OverallQual', 'GrLivArea', 'GarageArea', 'TotalBsmtSF', 'KitchenQual',
    'TotalSF', 'YearsSinceRemod', 'Age', 'OverallQual_GrLivArea', 
    '1stFlrSF', '2ndFlrSF', 'FullBath', 'ExterQual', 'BsmtQual', 
    'Neighborhood', 'MSZoning', 'LotFrontage' 
]

X_all = df_full_features[ROBUST_FEATURES]


# Identify feature types based on the robust set
# Use 'None' for object features where missing means 'None' (e.g., Basement quality)
# Use 'median' for features where missing is likely an error (e.g., LotFrontage)
numeric_features = X_all.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_all.select_dtypes(include=['object']).columns.tolist()


# Define the ColumnTransformer (Enhanced Imputation and Scaling)
preprocessor = ColumnTransformer(
    transformers=[
        # Numerical: Impute median (for errors), then Scale (StandardScaler)
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()) 
        ]), numeric_features),
        
        # Categorical: Impute 'missing' (for structural NA), then OneHotEncode
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')), 
            ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
        ]), categorical_features)
    ],
    remainder='passthrough'
)

# --- 6. SPLIT DATA ---
# Split the cleaned training set (using its index from the full processed set)
train_len_cleaned = len(X)
X_train_final = X_all.iloc[:train_len_cleaned]
X_test_final = X_all.iloc[train_len_cleaned:]

X_train, X_val, y_train, y_val = train_test_split(
    X_train_final, y, test_size=0.2, random_state=RANDOM_STATE
)

# --- 7. MODELING (Robust Pipelines) ---
print(f"--- ROBUST MODEL EVALUATION ({datetime.date.today().strftime('%d %B %Y')}) ---")


# A. Ridge Regression Pipeline (A powerful linear model for this dataset)
ridge_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=10.0, random_state=RANDOM_STATE)) # Use a known good alpha
])
ridge_pipe.fit(X_train, y_train)
ridge_rmse = np.sqrt(mean_squared_error(y_val, ridge_pipe.predict(X_val)))
print(f"Ridge RMSE (log): {ridge_rmse:.5f}")


# B. XGBoost Pipeline (Best single model benchmark)
# Increased estimators and deeper parameters for better fit
xgb_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.7,
        colsample_bytree=0.7,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=50,
        eval_metric='rmse'
    ))
])

# Process X_val separately for XGBoost's early stopping
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

# XGBoost is trained outside the final pipeline for early stopping validation
xgb_reg = xgb_pipe.named_steps['regressor']
xgb_reg.fit(
    X_train_processed, 
    y_train, 
    eval_set=[(X_val_processed, y_val)], 
    verbose=False
)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_reg.predict(X_val_processed)))
print(f"XGBoost RMSE (log): {xgb_rmse:.5f}")

print("\nEnhancement complete: Switched to robust features, scaling, and integrated pipelines.")

Analysis on 30 October 2025

🎯 Summary of Improvement (U2 to  U3)

The movement from U2 to U3 represents a common progression in a machine learning project:From Feature Selection to Feature Engineering: U3 moves from selecting a basic set of features (U2) to proactively creating more informative features (e.g., TotalSF, YearsSinceRemod, interaction terms) that often capture non-linear relationships and domain knowledge, leading to better model performance.

Richer Model Configuration: The XGBoost model in U3 uses a more comprehensive set of hyperparameters and explicitly leverages the eval_set and early_stopping_rounds for more robust training that helps prevent overfitting while achieving a better fit on the training data.

Consistent Preprocessing: U3 correctly applies feature engineering on a combined dataset to ensure the same logic and derived features exist for both training and test data, a vital step missed in U2's simpler approach.

In essence, Code U3 is a deeper, more specialized solution focused on squeezing more performance out of the dataset through advanced data manipulation and model tuning. Code U2 is a clearer, better-organized mid-level benchmark that provides solid comparative results across multiple model types.

In [ ]:
# U4 - FINAL RUGGED CODE: Ridge Tuning and Ensemble Prep 🏆

# Forked Code - Finalized for Masterclass Submission

# Date of Finalization: 31 October 2025

# --- 1. Import Libraries ---

import pandas as pd
import numpy as np
import os
import datetime
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# --- Configuration (Standard) ---
DATA_DIRECTORY = '/kaggle/input/house-prices-advanced-regression-techniques/'
TARGET = 'SalePrice'
RANDOM_STATE = 42

# --- 2. DATA LOADING & INITIAL PREP ---
try:
    df_train = pd.read_csv(DATA_DIRECTORY + 'train.csv')
    df_test = pd.read_csv(DATA_DIRECTORY + 'test.csv')
except FileNotFoundError:
    print("FATAL ERROR: Files not found. Check DATA_DIRECTORY path.")
    raise

# --- 3. TARGET & OUTLIER HANDLING ---
# Outlier removal based on GrLivArea, only on the training set
df_train_cleaned = df_train[df_train['GrLivArea'] < 4000].copy()
y = np.log(df_train_cleaned[TARGET]) # Log transform the target
X = df_train_cleaned.drop(TARGET, axis=1)

# Combine cleaned train features (X) and test features for unified preprocessing
df_full_cleaned = pd.concat([X, df_test], ignore_index=True)


# --- 4. FEATURE ENGINEERING (Advanced) ---

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Creates derived features essential for model performance."""
    df_eng = df.copy()

    # Total Square Footage
    df_eng['TotalSF'] = df_eng['TotalBsmtSF'] + df_eng['1stFlrSF'] + df_eng['2ndFlrSF']

    # Time-based features
    df_eng['YearsSinceRemod'] = df_eng['YrSold'] - df_eng['YearRemodAdd']
    df_eng['Age'] = df_eng['YrSold'] - df_eng['YearBuilt']
    df_eng.loc[df_eng['Age'] < 0, 'Age'] = 0

    # Interaction terms
    df_eng['OverallQual_GrLivArea'] = df_eng['OverallQual'] * df_eng['GrLivArea']
    df_eng['TotalSF_sq'] = df_eng['TotalSF'] ** 2 # MASTERCLASS TWEAK: Polynomial term

    # Binary features
    df_eng['HasBasement'] = (df_eng['TotalBsmtSF'] > 0).astype(int)
    df_eng['HasGarage'] = (df_eng['GarageArea'] > 0).astype(int)

    return df_eng

df_full_features = engineer_features(df_full_cleaned.drop('Id', axis=1))


# --- 5. RUGGED FEATURE SET & PREPROCESSING PIPELINE ---

# Expanded feature set including the new polynomial term
ROBUST_FEATURES = [
    'OverallQual', 'GrLivArea', 'GarageArea', 'TotalBsmtSF', 'KitchenQual',
    'TotalSF', 'YearsSinceRemod', 'Age', 'OverallQual_GrLivArea', 'TotalSF_sq',
    '1stFlrSF', '2ndFlrSF', 'FullBath', 'ExterQual', 'BsmtQual',
    'Neighborhood', 'MSZoning', 'LotFrontage'
]

X_all = df_full_features[ROBUST_FEATURES]

numeric_features = X_all.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_all.select_dtypes(include=['object']).columns.tolist()


# Define the ColumnTransformer for robust preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        # Numerical: Impute median, then Standard Scale
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),

        # Categorical: Impute 'Missing', then One-Hot Encode
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
        ]), categorical_features)
    ],
    remainder='passthrough'
)

# --- 6. SPLIT DATA ---
train_len_cleaned = len(X)
X_train_final = X_all.iloc[:train_len_cleaned]

# Split the cleaned training data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_final, y, test_size=0.2, random_state=RANDOM_STATE
)

# --- 7. MODELING (Tuning and Final Ensemble Prep) ---

print(f"--- RUGGED MODEL TUNING & ENSEMBLE PREP ({datetime.date.today().strftime('%d %B %Y')}) ---")


# A. Ridge Tuning via GridSearchCV
ridge_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(random_state=RANDOM_STATE, solver='lsqr'))
])

# Search a tight range for the optimal alpha
param_grid = {'regressor__alpha': np.arange(8.0, 12.1, 0.5).tolist()}

grid_search = GridSearchCV(
    ridge_pipe,
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_ridge = grid_search.best_estimator_
best_alpha = grid_search.best_params_['regressor__alpha']
ridge_rmse = np.sqrt(mean_squared_error(y_val, best_ridge.predict(X_val)))

print(f"1. OPTIMAL RIDGE RMSE (log): {ridge_rmse:.5f} (Alpha: {best_alpha:.2f})")


# B. XGBoost Benchmark (Corrected for successful pipeline fit)
xgb_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=1000, 
        learning_rate=0.05, 
        max_depth=4, 
        subsample=0.7,
        colsample_bytree=0.7, 
        random_state=RANDOM_STATE, 
        n_jobs=-1,
        # REMOVED: early_stopping_rounds=50, to allow for seamless Pipeline.fit()
        eval_metric='rmse'
    ))
])

# Fit XGBoost successfully
xgb_pipe.fit(X_train, y_train)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_pipe.predict(X_val)))
print(f"2. XGBoost RMSE (log): {xgb_rmse:.5f}")


# C. Ensemble Preparation Summary
print("\n--- NEXT STEP: ENSEMBLE (Weighted Average) ---")
print("Train Best Ridge (alpha={:.2f}) and XGBoost on ALL training data.".format(best_alpha))
print(f"Use a blend (e.g., 60% Ridge, 40% XGBoost) for the final submission.")

🎯 Summary of Improvements  -  U4

The U4 code represents a significant step towards a final, competitive solution by focusing on optimization and integration for ensemble modeling  

Model Optimization:
The most critical linear model, Ridge Regression, is rigorously tuned using GridSearchCV to find the mathematically optimal regularization parameter ($\alpha$).

Enhanced Feature Power: 
The addition of the squared total square footage (TotalSF_sq) is a powerful, well-known trick in this dataset, which significantly improves the Ridge model's fit by capturing non-linearity.

Masterclass Readiness: The code is structured to produce two highly optimized model outputs (Tuned Ridge and Robust XGBoost) and concludes with an explicit strategy (Ensemble/Blending) for the final submission, which is the standard procedure for top machine learning competition solutions.

In [ ]:
# U5 - FINAL RUGGED CODE: Ridge Tuning and Ensemble Prep (XGBoost Fixed) 🏆

# Forked Code - Finalized for Masterclass Submission

# Date of Finalization: 01 November 2025

# --- 1. Import Libraries ---

import pandas as pd
import numpy as np
import os
import datetime
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge 
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler 
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor 

# --- Configuration (Standard) ---
DATA_DIRECTORY = '/kaggle/input/house-prices-advanced-regression-techniques/'
TARGET = 'SalePrice'
RANDOM_STATE = 42 

# --- 2. DATA LOADING & INITIAL PREP ---
try:
    df_train = pd.read_csv(DATA_DIRECTORY + 'train.csv')
    df_test = pd.read_csv(DATA_DIRECTORY + 'test.csv') 
except FileNotFoundError:
    print("FATAL ERROR: Files not found. Check DATA_DIRECTORY path.")
    raise

# --- 3. TARGET & OUTLIER HANDLING ---
df_train_cleaned = df_train[df_train['GrLivArea'] < 4000].copy()
y = np.log(df_train_cleaned[TARGET])
X = df_train_cleaned.drop(TARGET, axis=1)

# Recalculate full_df based on the cleaned train set and the original test set
df_full_cleaned = pd.concat([X, df_test], ignore_index=True)


# --- 4. FEATURE ENGINEERING (Advanced) ---

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Creates derived features essential for linear model performance."""
    df_eng = df.copy()
    df_eng['TotalSF'] = df_eng['TotalBsmtSF'] + df_eng['1stFlrSF'] + df_eng['2ndFlrSF']
    df_eng['YearsSinceRemod'] = df_eng['YrSold'] - df_eng['YearRemodAdd']
    df_eng['Age'] = df_eng['YrSold'] - df_eng['YearBuilt']
    df_eng.loc[df_eng['Age'] < 0, 'Age'] = 0 
    df_eng['OverallQual_GrLivArea'] = df_eng['OverallQual'] * df_eng['GrLivArea']
    df_eng['HasBasement'] = (df_eng['TotalBsmtSF'] > 0).astype(int)
    df_eng['HasGarage'] = (df_eng['GarageArea'] > 0).astype(int)
    
    # MASTERCLASS TWEAK: Adding a known important polynomial/interaction term
    df_eng['TotalSF_sq'] = df_eng['TotalSF'] ** 2
    
    return df_eng

df_full_features = engineer_features(df_full_cleaned.drop('Id', axis=1))


# --- 5. RUGGED FEATURE SET & PREPROCESSING PIPELINE ---

ROBUST_FEATURES = [
    'OverallQual', 'GrLivArea', 'GarageArea', 'TotalBsmtSF', 'KitchenQual',
    'TotalSF', 'YearsSinceRemod', 'Age', 'OverallQual_GrLivArea', 'TotalSF_sq',
    '1stFlrSF', '2ndFlrSF', 'FullBath', 'ExterQual', 'BsmtQual', 
    'Neighborhood', 'MSZoning', 'LotFrontage' 
]

X_all = df_full_features[ROBUST_FEATURES]

numeric_features = X_all.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_all.select_dtypes(include=['object']).columns.tolist()


# Define the ColumnTransformer 
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()) 
        ]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')), 
            ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
        ]), categorical_features)
    ],
    remainder='passthrough'
)

# --- 6. SPLIT DATA ---
train_len_cleaned = len(X)
X_train_final = X_all.iloc[:train_len_cleaned]

X_train, X_val, y_train, y_val = train_test_split(
    X_train_final, y, test_size=0.2, random_state=RANDOM_STATE
)

# --- 7. MODELING (Tuning and Final Ensemble Prep) ---

print(f"--- RUGGED MODEL TUNING & ENSEMBLE PREP ({datetime.date.today().strftime('%d %B %Y')}) ---")

# A. Ridge Tuning (Use the results from the successful run)
best_alpha = 8.00 # Use the optimal alpha found previously to skip re-tuning
best_ridge = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=best_alpha, random_state=RANDOM_STATE, solver='lsqr')) 
])
# Fit the full Ridge pipeline
best_ridge.fit(X_train, y_train)
ridge_rmse = np.sqrt(mean_squared_error(y_val, best_ridge.predict(X_val)))

print(f"1. OPTIMAL RIDGE RMSE (log): {ridge_rmse:.5f} (Alpha: {best_alpha:.2f})")

# B. XGBoost Benchmark (ERROR-FREE FIX: Training outside the full Pipeline)

# Define the XGBoost Regressor (no pipeline needed here, just the model)
xgb_regressor = XGBRegressor(
    n_estimators=1000, learning_rate=0.05, max_depth=4, subsample=0.7, 
    colsample_bytree=0.7, random_state=RANDOM_STATE, n_jobs=-1,
    early_stopping_rounds=50, eval_metric='rmse'
)

# 1. Transform the Data Manually (This resolves the ValueError)
# Clone the preprocessor to ensure it's a fresh fit for this block
preprocessor_fit = preprocessor.fit(X_train) 

# Transform X_train (to learn scaling/encoding)
X_train_processed = preprocessor_fit.transform(X_train)
# Transform X_val (to apply learned scaling/encoding)
X_val_processed = preprocessor_fit.transform(X_val)

# 2. Fit the Regressor Manually (with early stopping on processed data)
xgb_regressor.fit(
    X_train_processed, 
    y_train, 
    eval_set=[(X_val_processed, y_val)], # Now the eval_set has numerical data
    verbose=False 
)

xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_regressor.predict(X_val_processed)))
print(f"2. XGBoost RMSE (log): {xgb_rmse:.5f}")


# C. Ensemble Preparation Summary
print("\n--- NEXT STEP: ENSEMBLE (Weighted Average) ---")
print(f"Final Step: Train Best Ridge (alpha={best_alpha:.2f}) and XGBoost on ALL training data.")
print("Use a blend (e.g., 60% Ridge, 40% XGBoost) for the final submission to secure a medal!")

📝 Summary of Evolution : 

01 November 2025

The evolution from U3 to U5 demonstrates a standard progression in a machine learning project:

U3 (Robustness): Focus is on creating solid preprocessing pipelines and establishing initial model benchmarks (Ridge, XGBoost).

U4 (Optimization): Focus shifts to maximizing model performance by tuning the linear model (Ridge) hyperparameter Alpha using GridSearchCV and introducing a powerful new non-linear feature (TotalSF_sq).

U5 (Finalization and Stability): The code locks in the best performing Alpya value and critically fixes the technical error in the XGBoost pipeline that prevented correct early stopping, ensuring the entire process is robust and ready for the final ensemble prediction.

In [ ]:
# U6 - FINAL RUGGED CODE: XGBoost Evaluation Fixed (Ready for Ensemble)

# Forked Code - Finalized for Masterclass Submission

# Date of Finalization: 02 November 2025

# --- 1. Import Libraries (Same as before) ---

import pandas as pd
import numpy as np
import os
import datetime
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge 
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler 
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor 

# --- Configuration (Standard) ---
DATA_DIRECTORY = '/kaggle/input/house-prices-advanced-regression-techniques/'
TARGET = 'SalePrice'
RANDOM_STATE = 42 

# --- 2. DATA LOADING & INITIAL PREP ---
try:
    df_train = pd.read_csv(DATA_DIRECTORY + 'train.csv')
    df_test = pd.read_csv(DATA_DIRECTORY + 'test.csv') 
except FileNotFoundError:
    print("FATAL ERROR: Files not found. Check DATA_DIRECTORY path.")
    raise

# --- 3. TARGET & OUTLIER HANDLING ---
df_train_cleaned = df_train[df_train['GrLivArea'] < 4000].copy()
y = np.log(df_train_cleaned[TARGET])
X = df_train_cleaned.drop(TARGET, axis=1)

df_full_cleaned = pd.concat([X, df_test], ignore_index=True)


# --- 4. FEATURE ENGINEERING (Advanced) ---

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Creates derived features essential for linear model performance."""
    df_eng = df.copy()
    df_eng['TotalSF'] = df_eng['TotalBsmtSF'] + df_eng['1stFlrSF'] + df_eng['2ndFlrSF']
    df_eng['YearsSinceRemod'] = df_eng['YrSold'] - df_eng['YearRemodAdd']
    df_eng['Age'] = df_eng['YrSold'] - df_eng['YearBuilt']
    df_eng.loc[df_eng['Age'] < 0, 'Age'] = 0 
    df_eng['OverallQual_GrLivArea'] = df_eng['OverallQual'] * df_eng['GrLivArea']
    df_eng['HasBasement'] = (df_eng['TotalBsmtSF'] > 0).astype(int)
    df_eng['HasGarage'] = (df_eng['GarageArea'] > 0).astype(int)
    df_eng['TotalSF_sq'] = df_eng['TotalSF'] ** 2
    
    return df_eng

df_full_features = engineer_features(df_full_cleaned.drop('Id', axis=1))


# --- 5. RUGGED FEATURE SET & PREPROCESSING PIPELINE ---

ROBUST_FEATURES = [
    'OverallQual', 'GrLivArea', 'GarageArea', 'TotalBsmtSF', 'KitchenQual',
    'TotalSF', 'YearsSinceRemod', 'Age', 'OverallQual_GrLivArea', 'TotalSF_sq',
    '1stFlrSF', '2ndFlrSF', 'FullBath', 'ExterQual', 'BsmtQual', 
    'Neighborhood', 'MSZoning', 'LotFrontage' 
]

X_all = df_full_features[ROBUST_FEATURES]

numeric_features = X_all.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_all.select_dtypes(include=['object']).columns.tolist()

# Define the ColumnTransformer (Pre-processor)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()) 
        ]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')), 
            ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
        ]), categorical_features)
    ],
    remainder='passthrough'
)

# --- 6. SPLIT DATA ---
train_len_cleaned = len(X)
X_train_final = X_all.iloc[:train_len_cleaned]

X_train, X_val, y_train, y_val = train_test_split(
    X_train_final, y, test_size=0.2, random_state=RANDOM_STATE
)

# --- 7. MODELING (Tuning and Final Ensemble Prep) ---

print(f"--- RUGGED MODEL TUNING & ENSEMBLE PREP (24 October 2025) ---")

# A. Ridge Tuning (Skip GridSearch and use the optimal alpha found: 8.00)
best_alpha = 8.00 
best_ridge = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=best_alpha, random_state=RANDOM_STATE, solver='lsqr')) 
])
best_ridge.fit(X_train, y_train)
ridge_rmse = np.sqrt(mean_squared_error(y_val, best_ridge.predict(X_val)))

print(f"1. OPTIMAL RIDGE RMSE (log): {ridge_rmse:.5f} (Alpha: {best_alpha:.2f})")

# B. XGBoost Benchmark (FIXED for early stopping)
xgb_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=1000, learning_rate=0.05, max_depth=4, subsample=0.7, 
        colsample_bytree=0.7, random_state=RANDOM_STATE, n_jobs=-1,
        early_stopping_rounds=50, eval_metric='rmse'
    ))
])

# 1. Transform X_train and X_val using the preprocessor step ONLY
# We must fit the preprocessor first to ensure the feature space is consistent.
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

# 2. Extract the bare XGBRegressor from the pipe and fit it manually 
# This is the standard, reliable way to use early_stopping_rounds with Pipelines.
xgb_reg_final = xgb_pipe.named_steps['regressor']

# 3. Fit the bare regressor using the manually transformed data
xgb_reg_final.fit(
    X_train_processed, 
    y_train, 
    eval_set=[(X_val_processed, y_val)], # FIX: Pass the pre-processed data
    verbose=False
)

xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_reg_final.predict(X_val_processed)))
print(f"2. XGBoost RMSE (log): {xgb_rmse:.5f}")


# C. Ensemble Preparation Summary
print("\n--- FINAL STEP: ENSEMBLE ---")
print("Ridge is ready: RMSE {:.5f}".format(ridge_rmse))
print("XGBoost is ready: RMSE {:.5f}".format(xgb_rmse))
print("The final submission should blend predictions from these two models (e.g., 60% Ridge, 40% XGBoost) after training on the full X_train_final.")

🧱 Explanation of the Code Structure and Steps :

Data Preparation and Outlier HandlingTarget Transformation:

The target variable, SalePrice, is log-transformed. This is standard practice in regression, especially with monetary values, as it helps normalize the skewed distribution and prevents large prices from dominating the loss function.

Outlier Removal: 

An outlier is removed by filtering houses where the GrLivArea (Above ground living area) is less than 4000 sq ft.Data Combination: Training and test data are combined into df_full_cleaned to ensure consistent feature engineering and imputation across both sets.

 Feature Engineering (Section 4)
 
 A set of essential derived features are created to improve model performance, particularly for the linear model (Ridge)
 
TotalSF: 

Sum of basement, 1st, and 2nd floor square footage (a key predictor).
YearsSinceRemod / Age: Time-based features to capture the house's life cycle.

OverallQual_GrLivArea: 

An interaction feature that captures the multiplicative effect of quality and size.

TotalSF_sq: 

A non-linear term ($\text{TotalSF}^2$) for the linear model to better fit non-linear data.

Preprocessing Pipeline (Section 5)

A single, powerful ColumnTransformer is defined to handle different feature types robustly:

Feature Selection (ROBUST_FEATURES): 

Only a curated list of features is used, reducing noise.

Numerical Pipeline: 

Fills missing values with the median (SimpleImputer) and then scales the data using StandardScaler. Scaling is critical for Ridge Regression (linear models).

Categorical Pipeline: Fills missing values with the constant string 'Missing' and then applies OneHotEncoder to convert categories into numerical features.

 Model Training & Ensemble Preparation (Section 7)
 
 The core logic trains the two ensemble components and evaluates them on a validation set (X_val).

 Ridge Regression
 
 Hyperparameter:
 
 The optimal Alpha (regularization strength) is pre-set to 8.00, skipping a resource-intensive GridSearchCV.
 
 Training: 
 
 The model is trained using the full Scikit-learn Pipeline, which correctly applies all preprocessing steps before fitting the Ridge regressor.
 
 Validation:
 
 It achieves an text{RMSE}_{\log} of 0.14145 on the validation set.

 XGBoost Regressor 
 (The "FIX")This section implements a workaround to use Early Stopping effectively:
 
 Issue: 
 
 The native Pipeline.fit() in older Scikit-learn versions doesn't cleanly pass the eval_set (validation data) required by XGBoost for early stopping.
 
 Solution:
 
 The preprocessor is manually fitted and used to transform X_train and X_val into NumPy arrays (X_train_processed, X_val_processed).The bare XGBRegressor object is extracted (xgb_reg_final).

 The regressor is fitted manually using the transformed data and, crucially, the eval_set=[(X_val_processed, y_val)] argument. This allows the model to stop training after 50 rounds (early_stopping_rounds=50) if no improvement is seen on the validation set, preventing overfitting.Validation: It achieves an text{RMSE}_{\log} of 0.15018 on the validation set.

In [ ]:
# U7

# U7 - MASTERCLASS FINAL SUBMISSION: Ensemble Blending

# Date of Submission: 03 November 2025

# --- 1. Import Libraries ---
import pandas as pd
import numpy as np
import os
import datetime
from sklearn.linear_model import Ridge 
from sklearn.preprocessing import StandardScaler, OneHotEncoder 
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor 

import warnings
# Only ignore warnings within this block of code
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    # Code that generates the warning goes here

# --- Configuration & Optimal Parameters ---
DATA_DIRECTORY = '/kaggle/input/house-prices-advanced-regression-techniques/'
TARGET = 'SalePrice'
RANDOM_STATE = 42 

# Optimal parameters determined in the validation phase
OPTIMAL_RIDGE_ALPHA = 8.00
RIDGE_WEIGHT = 0.60 # Based on lower RMSE (0.14145)
XGB_WEIGHT = 0.40   # Based on higher RMSE (0.15018)

# --- 2. DATA LOADING & PREP (Final Full Set) ---
try:
    df_train = pd.read_csv(DATA_DIRECTORY + 'train.csv')
    df_test = pd.read_csv(DATA_DIRECTORY + 'test.csv') 
except FileNotFoundError:
    print("FATAL ERROR: Files not found.")
    raise

# Store Test IDs for submission file
test_ids = df_test['Id']

# --- 3. TARGET, OUTLIER HANDLING, and SPLIT ---
df_train_cleaned = df_train[df_train['GrLivArea'] < 4000].copy()
y_full = np.log(df_train_cleaned[TARGET]) # Use y_full for final training
X_train_full = df_train_cleaned.drop(TARGET, axis=1)

df_full_cleaned = pd.concat([X_train_full, df_test], ignore_index=True)


# --- 4. FEATURE ENGINEERING (Advanced) ---

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Creates derived features essential for the ensemble models."""
    df_eng = df.copy()
    df_eng['TotalSF'] = df_eng['TotalBsmtSF'] + df_eng['1stFlrSF'] + df_eng['2ndFlrSF']
    df_eng['YearsSinceRemod'] = df_eng['YrSold'] - df_eng['YearRemodAdd']
    df_eng['Age'] = df_eng['YrSold'] - df_eng['YearBuilt']
    df_eng.loc[df_eng['Age'] < 0, 'Age'] = 0 
    df_eng['OverallQual_GrLivArea'] = df_eng['OverallQual'] * df_eng['GrLivArea']
    df_eng['TotalSF_sq'] = df_eng['TotalSF'] ** 2
    return df_eng

df_full_features = engineer_features(df_full_cleaned.drop('Id', axis=1))


# --- 5. RUGGED FEATURE SET & PREPROCESSING (Re-used) ---

ROBUST_FEATURES = [
    'OverallQual', 'GrLivArea', 'GarageArea', 'TotalBsmtSF', 'KitchenQual',
    'TotalSF', 'YearsSinceRemod', 'Age', 'OverallQual_GrLivArea', 'TotalSF_sq',
    '1stFlrSF', '2ndFlrSF', 'FullBath', 'ExterQual', 'BsmtQual', 
    'Neighborhood', 'MSZoning', 'LotFrontage' 
]

X_all = df_full_features[ROBUST_FEATURES]

numeric_features = X_all.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_all.select_dtypes(include=['object']).columns.tolist()

# Define the ColumnTransformer (Pre-processor)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='constant', fill_value='Missing')), 
                                ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))]), categorical_features)
    ],
    remainder='passthrough'
)

# Split data back into final training and testing features
train_len_cleaned = len(X_train_full)
X_train_final = X_all.iloc[:train_len_cleaned]
X_test_final = X_all.iloc[train_len_cleaned:]


# --- 6. FINAL MODEL TRAINING AND PREDICTION ---

print(f"--- FINAL ENSEMBLE TRAINING ({datetime.date.today().strftime('%d %B %Y')}) ---")

# A. Ridge Model Training (Trained on full data)
ridge_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=OPTIMAL_RIDGE_ALPHA, random_state=RANDOM_STATE, solver='lsqr')) 
])
ridge_model.fit(X_train_final, y_full)
ridge_preds_log = ridge_model.predict(X_test_final)
print(f"1. Ridge Trained (Alpha: {OPTIMAL_RIDGE_ALPHA:.2f}). Predictions generated.")

# B. XGBoost Model Training (Trained on full data)
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=1000, learning_rate=0.05, max_depth=4, subsample=0.7, 
        colsample_bytree=0.7, random_state=RANDOM_STATE, n_jobs=-1,
        # Early stopping is often removed/increased for final training on full data
        # to ensure the model uses all data. We'll stick to a simple fit here.
    ))
])
xgb_model.fit(X_train_final, y_full)
xgb_preds_log = xgb_model.predict(X_test_final)
print(f"2. XGBoost Trained. Predictions generated.")


# --- 7. ENSEMBLE BLENDING and Inverse Transformation ---

# Weighted average of log predictions
ensemble_preds_log = (RIDGE_WEIGHT * ridge_preds_log) + (XGB_WEIGHT * xgb_preds_log)

# Inverse log transformation (Crucial step)
ensemble_preds = np.expm1(ensemble_preds_log) 

# Safety check for negative prices
ensemble_preds[ensemble_preds < 0] = 0

print(f"\n3. Ensemble Blend: {RIDGE_WEIGHT*100}% Ridge + {XGB_WEIGHT*100}% XGBoost.")


# --- 8. GENERATE SUBMISSION FILE ---

submission_df = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': ensemble_preds
})

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission_file_name = f'submission_U1_ENSEMBLE_RIDGE_{RIDGE_WEIGHT*100}pc_{timestamp}.csv' 
submission_df.to_csv(submission_file_name, index=False)

print(f"\n🎉 SUCCESS: Final masterclass submission '{submission_file_name}' ready.")

In [ ]:
# U8 - MASTERCLASS FINAL SUBMISSION: Ensemble Blending - FIXED

# Date of Submission: 04 November 2025

# --- 1. Import Libraries ---
import pandas as pd
import numpy as np
import os
import datetime
from sklearn.linear_model import Ridge 
from sklearn.preprocessing import StandardScaler, OneHotEncoder 
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor 

import warnings # Keep import warnings

# --- Configuration & Optimal Parameters ---
DATA_DIRECTORY = '/kaggle/input/house-prices-advanced-regression-techniques/'
TARGET = 'SalePrice'
RANDOM_STATE = 42 

# Optimal parameters determined in the validation phase
OPTIMAL_RIDGE_ALPHA = 8.00
RIDGE_WEIGHT = 0.60 # Based on lower RMSE (0.14145)
XGB_WEIGHT = 0.40   # Based on higher RMSE (0.15018)

# --- 2. DATA LOADING & PREP (Final Full Set) ---
try:
    df_train = pd.read_csv(DATA_DIRECTORY + 'train.csv')
    df_test = pd.read_csv(DATA_DIRECTORY + 'test.csv') 
except FileNotFoundError:
    print("FATAL ERROR: Files not found.")
    raise

# Store Test IDs for submission file
test_ids = df_test['Id']

# --- 3. TARGET, OUTLIER HANDLING, and SPLIT ---
df_train_cleaned = df_train[df_train['GrLivArea'] < 4000].copy()
y_full = np.log(df_train_cleaned[TARGET]) 
X_train_full = df_train_cleaned.drop(TARGET, axis=1)

df_full_cleaned = pd.concat([X_train_full, df_test], ignore_index=True)


# --- 4. FEATURE ENGINEERING (Advanced) ---

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Creates derived features essential for the ensemble models."""
    df_eng = df.copy()
    df_eng['TotalSF'] = df_eng['TotalBsmtSF'] + df_eng['1stFlrSF'] + df_eng['2ndFlrSF']
    df_eng['YearsSinceRemod'] = df_eng['YrSold'] - df_eng['YearRemodAdd']
    df_eng['Age'] = df_eng['YrSold'] - df_eng['YearBuilt']
    df_eng.loc[df_eng['Age'] < 0, 'Age'] = 0 
    df_eng['OverallQual_GrLivArea'] = df_eng['OverallQual'] * df_eng['GrLivArea']
    df_eng['TotalSF_sq'] = df_eng['TotalSF'] ** 2
    return df_eng

df_full_features = engineer_features(df_full_cleaned.drop('Id', axis=1))


# --- 5. RUGGED FEATURE SET & PREPROCESSING (Re-used) ---

ROBUST_FEATURES = [
    'OverallQual', 'GrLivArea', 'GarageArea', 'TotalBsmtSF', 'KitchenQual',
    'TotalSF', 'YearsSinceRemod', 'Age', 'OverallQual_GrLivArea', 'TotalSF_sq',
    '1stFlrSF', '2ndFlrSF', 'FullBath', 'ExterQual', 'BsmtQual', 
    'Neighborhood', 'MSZoning', 'LotFrontage' 
]

X_all = df_full_features[ROBUST_FEATURES]

numeric_features = X_all.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_all.select_dtypes(include=['object']).columns.tolist()

# Define the ColumnTransformer (Pre-processor)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='constant', fill_value='Missing')), 
                                ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))]), categorical_features)
    ],
    remainder='passthrough'
)

# Split data back into final training and testing features
train_len_cleaned = len(X_train_full)
X_train_final = X_all.iloc[:train_len_cleaned]
X_test_final = X_all.iloc[train_len_cleaned:]


# --- 6. FINAL MODEL TRAINING AND PREDICTION ---

print(f"--- FINAL ENSEMBLE TRAINING ({datetime.date.today().strftime('%d %B %Y')}) ---")

# A. Define Models
ridge_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=OPTIMAL_RIDGE_ALPHA, random_state=RANDOM_STATE, solver='lsqr')) 
])
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=1000, learning_rate=0.05, max_depth=4, subsample=0.7, 
        colsample_bytree=0.7, random_state=RANDOM_STATE, n_jobs=-1
    ))
])

# FIX: Wrap the fitting and prediction in the professional warning suppression block
with warnings.catch_warnings():
    warnings.filterwarnings("ignore") # Ignore all warnings within this block

    # B. Ridge Model Training 
    ridge_model.fit(X_train_final, y_full)
    ridge_preds_log = ridge_model.predict(X_test_final)
    print(f"1. Ridge Trained (Alpha: {OPTIMAL_RIDGE_ALPHA:.2f}). Predictions generated.")

    # C. XGBoost Model Training
    xgb_model.fit(X_train_final, y_full)
    xgb_preds_log = xgb_model.predict(X_test_final)
    print(f"2. XGBoost Trained. Predictions generated.")


# --- 7. ENSEMBLE BLENDING and Inverse Transformation ---

# Weighted average of log predictions
ensemble_preds_log = (RIDGE_WEIGHT * ridge_preds_log) + (XGB_WEIGHT * xgb_preds_log)

# Inverse log transformation 
ensemble_preds = np.expm1(ensemble_preds_log) 

# Safety check for negative prices
ensemble_preds[ensemble_preds < 0] = 0

print(f"\n3. Ensemble Blend: {RIDGE_WEIGHT*100}% Ridge + {XGB_WEIGHT*100}% XGBoost.")


# --- 8. GENERATE SUBMISSION FILE ---

submission_df = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': ensemble_preds
})

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission_file_name = f'submission_U7_ENSEMBLE_RIDGE_{RIDGE_WEIGHT*100}pc_{timestamp}.csv' 
submission_df.to_csv(submission_file_name, index=False)

print(f"\n🎉 SUCCESS: Final masterclass submission '{submission_file_name}' ready. (Warnings suppressed professionally.)")